# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [4]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    for attempt in range(1, tries + 1):
        try:
            r = requests.get(BASE, params=params, timeout=10)

            if r.status_code == 200:
                return pd.DataFrame(r.json()['daily'])

            if r.status_code in {400, 404}:
                r.raise_for_status()

            if r.status_code in RETRY_STATUSES and attempt < tries:
                time.sleep(2 ** attempt)
                continue

            r.raise_for_status()

        except requests.exceptions.RequestException as e:
            if attempt == tries:
                raise e
            time.sleep(2 ** attempt)

    raise RuntimeError(f"Failed to fetch weather after {tries} attempts.")

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [5]:
used_fallback = False

try:
    wx = get_weather(start='2026-09-05', end='2026-10-03')
    wx['time'] = pd.to_datetime(wx['time']).dt.strftime('%Y-%m-%d')
    wx = wx[wx['time'].isin(['2026-09-05', '2026-10-03'])].reset_index(drop=True)
except Exception:
    used_fallback = True
    wx = FALLBACK.copy()

### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [6]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

if used_fallback:
    wx = FALLBACK.copy()

wx['source'] = 'fallback sample' if used_fallback else 'open-meteo'

### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [7]:
wx = wx.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm'
})

wx['date'] = pd.to_datetime(wx['date'])

### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [8]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [9]:
row_oct = wx[wx['date'] == '2026-10-03'].iloc[0]
row_sep = wx[wx['date'] == '2026-09-05'].iloc[0]

diff = row_oct['precip_mm'] - row_sep['precip_mm']
print(f"On October 3, 2026, it was wetter than September 5, 2026, with {row_oct['precip_mm']} mm of precipitation compared to {row_sep['precip_mm']} mm—making October 3 wetter by {diff:.1f} mm.")

On October 3, 2026, it was wetter than September 5, 2026, with 14.7 mm of precipitation compared to 2.5 mm—making October 3 wetter by 12.2 mm.


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [10]:
import datetime

print(f"Endpoint: {BASE}")
print(f"Coordinates: Lat 38.03, Lon -78.51")
print(f"Date Range: 2026-09-05 to 2026-10-03")
print(f"Pulled At: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Data Source: {'Fallback sample' if used_fallback else 'Open-Meteo Archive API'}")

Endpoint: https://archive-api.open-meteo.com/v1/archive
Coordinates: Lat 38.03, Lon -78.51
Date Range: 2026-09-05 to 2026-10-03
Pulled At: 2026-10-11 22:16:42
Data Source: Open-Meteo Archive API


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

If get_weather returned an empty DataFrame on failure instead of raising an exception, pandas would execute an inner or left join against 200 days of sales without throwing an error. Because the weather data would be completely empty or missing for those keys, the resulting merged table would drop matching rows or fill the weather columns with null/NaN values. When you chart rain against revenue, visualization would silently omit those data points or show flatlined/missing trends. Nobody would catch it because the code completed successfully with a "valid" empty frame, masking the fact that the API call actually failed silently behind the scenes.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [11]:
def get_weather_location(start, end, lat, lon):
    df = get_weather(start, end, lat=lat, lon=lon)
    df['time'] = pd.to_datetime(df['time']).dt.strftime('%Y-%m-%d')
    df = df[df['time'].isin(['2026-09-05', '2026-10-03'])].reset_index(drop=True)
    return df

try:
    charlottesville = get_weather_location('2026-09-05', '2026-10-03', 38.03, -78.51)
    charlottesville['city'] = 'Charlottesville'

    orlando = get_weather_location('2026-09-05', '2026-10-03', 28.54, -81.38)
    orlando['city'] = 'Orlando'

    combined_wx = pd.concat([charlottesville, orlando], ignore_index=True)
    combined_wx = combined_wx.rename(columns={
        'time': 'date',
        'temperature_2m_max': 'temp_max',
        'precipitation_sum': 'precip_mm'
    })
    combined_wx['date'] = pd.to_datetime(combined_wx['date'])
    combined_wx['source'] = 'open-meteo'
except Exception:
    pass

combined_wx

,date,temp_max,precip_mm,city,source
0,2026-09-05,29.4,2.5,Charlottesville,open-meteo
1,2026-10-03,22.0,14.7,Charlottesville,open-meteo
2,2026-09-05,32.1,3.9,Orlando,open-meteo
3,2026-10-03,30.4,14.5,Orlando,open-meteo
